<a href="https://colab.research.google.com/github/prince127-web/GenAI/blob/main/EmotionAI_Emotionally_Intelligent_Agentic_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U openai gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 36.8 MB/s eta 0:00:00


In [3]:
from google.colab import userdata
from openai import OpenAI

# Get API key from Google Colab Secrets
OPENAI_API_KEY = userdata.get("openai")

# Create OpenAI client
client = OpenAI(api_key=OPENAI_API_KEY)

# Model
MODEL = "gpt-5.6-luna"

print("===================================")
print("      EmotionAI - AI Connection")
print("===================================")
print("OpenAI API connected successfully!")
print("Model:", MODEL)

      EmotionAI - AI Connection
OpenAI API connected successfully!
Model: gpt-5.6-luna


In [4]:
response = client.responses.create(
    model=MODEL,
    input="Say 'EmotionAI connection successful!'"
)

print(response.output_text)

EmotionAI connection successful!


In [6]:
import json
import re
import gradio as gr

In [7]:
def clean_json(text):

    text = text.strip()

    # Remove markdown code blocks
    text = re.sub(r"```json\s*", "", text)
    text = re.sub(r"```\s*", "", text)

    # Find JSON object
    start = text.find("{")
    end = text.rfind("}")

    if start != -1 and end != -1:
        text = text[start:end + 1]

    return json.loads(text)

In [8]:
def emotion_agent(user_message):

    instructions = """
You are the Emotion Detection Agent in an emotionally intelligent
agentic AI system.

Analyze the user's message and identify the most likely emotional state.

Allowed emotions:

Happy
Sad
Angry
Anxious
Excited
Frustrated
Confused
Neutral

Return ONLY valid JSON in this exact format:

{
    "emotion": "Happy",
    "confidence": 0.95,
    "evidence": "Short explanation"
}

Rules:
- confidence must be between 0 and 1
- do not diagnose mental health conditions
- focus only on the emotion expressed in the message
"""

    response = client.responses.create(
        model=MODEL,
        instructions=instructions,
        input=user_message
    )

    return clean_json(response.output_text)

In [9]:
def intent_agent(user_message):

    instructions = """
You are the Intent Detection Agent.

Determine what the user wants from the conversation.

Possible intents include:

- Seeking advice
- Seeking emotional support
- Asking a question
- Sharing good news
- Sharing a problem
- Requesting information
- General conversation
- Seeking motivation
- Seeking reassurance
- Requesting explanation

Return ONLY valid JSON:

{
    "intent": "Seeking advice",
    "goal": "Short explanation of what the user wants"
}
"""

    response = client.responses.create(
        model=MODEL,
        instructions=instructions,
        input=user_message
    )

    return clean_json(response.output_text)

In [10]:
def situation_agent(user_message):

    instructions = """
You are the Situation Analysis Agent.

Analyze the context of the user's message.

Determine:

1. What is happening?
2. What does the user need?
3. How important is the situation?

Priority can be:

low
medium
high

Return ONLY valid JSON:

{
    "situation": "Short description",
    "user_need": "What the user needs",
    "priority": "medium"
}
"""

    response = client.responses.create(
        model=MODEL,
        instructions=instructions,
        input=user_message
    )

    return clean_json(response.output_text)

In [11]:
def tone_agent(emotion, intent, situation):

    instructions = """
You are the Tone Decision Agent.

You decide how the final AI response should communicate with the user.

Available tones:

- Empathetic
- Encouraging
- Calm
- Professional
- Enthusiastic
- Reassuring
- Friendly
- Informative

Analyze the emotion, intent and situation.

Return ONLY valid JSON:

{
    "tone": "Calm and Encouraging",
    "style": "Supportive",
    "reason": "Short explanation"
}
"""

    context = f"""
EMOTION:
{json.dumps(emotion, indent=2)}

INTENT:
{json.dumps(intent, indent=2)}

SITUATION:
{json.dumps(situation, indent=2)}
"""

    response = client.responses.create(
        model=MODEL,
        instructions=instructions,
        input=context
    )

    return clean_json(response.output_text)

In [12]:
def response_agent(
    user_message,
    emotion,
    intent,
    situation,
    tone
):

    instructions = """
You are the Response Generation Agent of EmotionAI.

Generate the final response to the user.

You must:

1. Understand the user's message.
2. Consider the detected emotion.
3. Consider the user's intent.
4. Consider the situation.
5. Follow the selected tone.
6. Give a useful and natural response.
7. Avoid sounding robotic.
8. Do not mention internal agents.
9. Do not diagnose medical or psychological conditions.

Response behavior:

If the user is HAPPY:
Celebrate their success naturally.

If the user is EXCITED:
Match their positive energy.

If the user is SAD:
Be empathetic and supportive.

If the user is ANXIOUS:
Be calm, reassuring and practical.

If the user is FRUSTRATED:
Acknowledge the frustration and offer a solution.

If the user is CONFUSED:
Explain clearly and simply.

If the user is ANGRY:
Remain calm, respectful and solution-oriented.

If the user is NEUTRAL:
Give a helpful and informative response.

Return ONLY the final response text.
"""

    context = f"""
USER MESSAGE:
{user_message}

DETECTED EMOTION:
{json.dumps(emotion, indent=2)}

USER INTENT:
{json.dumps(intent, indent=2)}

SITUATION:
{json.dumps(situation, indent=2)}

SELECTED TONE:
{json.dumps(tone, indent=2)}
"""

    response = client.responses.create(
        model=MODEL,
        instructions=instructions,
        input=context
    )

    return response.output_text.strip()

In [13]:
def emotion_ai(user_message):

    # -----------------------------
    # AGENT 1: Emotion Detection
    # -----------------------------

    emotion = emotion_agent(user_message)

    # -----------------------------
    # AGENT 2: Intent Detection
    # -----------------------------

    intent = intent_agent(user_message)

    # -----------------------------
    # AGENT 3: Situation Analysis
    # -----------------------------

    situation = situation_agent(user_message)

    # -----------------------------
    # AGENT 4: Tone Decision
    # -----------------------------

    tone = tone_agent(
        emotion,
        intent,
        situation
    )

    # -----------------------------
    # AGENT 5: Response Generation
    # -----------------------------

    final_response = response_agent(
        user_message,
        emotion,
        intent,
        situation,
        tone
    )

    # Return complete analysis

    return {
        "emotion": emotion,
        "intent": intent,
        "situation": situation,
        "tone": tone,
        "response": final_response
    }

In [14]:
result = emotion_ai(
    "I have an interview tomorrow and I am really nervous. "
    "I have prepared a lot but I am still scared."
)

print(json.dumps(result, indent=4))

{
    "emotion": {
        "emotion": "Anxious",
        "confidence": 0.98,
        "evidence": "The user explicitly says they are nervous and scared about an interview tomorrow, despite preparing extensively."
    },
    "intent": {
        "intent": "Seeking emotional support",
        "goal": "The user wants reassurance and encouragement to cope with nervousness about their upcoming interview."
    },
    "situation": {
        "situation": "The user has a job interview tomorrow and is experiencing significant anxiety despite thorough preparation.",
        "user_need": "Reassurance and practical strategies to manage nerves and feel confident before and during the interview.",
        "priority": "medium"
    },
    "tone": {
        "tone": "Calm and Encouraging",
        "style": "Supportive",
        "reason": "The user is anxious about an upcoming interview and needs reassurance, confidence-building encouragement, and practical support."
    },
    "response": "It\u2019s comple

In [15]:
def analyze_and_respond(message):

    if not message or not message.strip():

        return (
            "Please enter a message.",
            "—",
            "—",
            "—",
            "—",
            "—"
        )

    try:

        result = emotion_ai(message)

        emotion = result["emotion"]
        intent = result["intent"]
        situation = result["situation"]
        tone = result["tone"]
        response = result["response"]

        return (
            response,

            emotion.get("emotion", "—"),

            f"{round(emotion.get('confidence', 0) * 100)}%",

            intent.get("intent", "—"),

            situation.get("situation", "—"),

            tone.get("tone", "—")
        )

    except Exception as e:

        return (
            f"Error: {str(e)}",
            "—",
            "—",
            "—",
            "—",
            "—"
        )

In [16]:
custom_css = """

body {
    background: #f4f7fb;
}

.gradio-container {
    max-width: 1150px !important;
}

.title {
    text-align: center;
    padding: 20px;
}

.title h1 {
    font-size: 40px;
    font-weight: 700;
    margin-bottom: 5px;
}

.title p {
    color: #667085;
    font-size: 16px;
}

"""

with gr.Blocks(
    title="EmotionAI",
    css=custom_css
) as demo:

    # --------------------------------
    # HEADER
    # --------------------------------

    gr.HTML(
        """
        <div class="title">

            <h1>EmotionAI</h1>

            <p>
            Emotionally Intelligent Agentic AI
            </p>

            <p>
            Understand • Analyze • Adapt • Respond
            </p>

        </div>
        """
    )

    # --------------------------------
    # MAIN LAYOUT
    # --------------------------------

    with gr.Row():

        # LEFT SIDE

        with gr.Column(scale=2):

            user_input = gr.Textbox(
                label="Your Message",
                placeholder=(
                    "Example: I have an interview tomorrow "
                    "and I'm really nervous..."
                ),
                lines=6
            )

            analyze_button = gr.Button(
                "Analyze & Respond",
                variant="primary"
            )

            response_output = gr.Textbox(
                label="AI Response",
                lines=8
            )

        # RIGHT SIDE

        with gr.Column(scale=1):

            gr.Markdown(
                "### Agent Analysis"
            )

            emotion_output = gr.Textbox(
                label="Detected Emotion"
            )

            confidence_output = gr.Textbox(
                label="Confidence"
            )

            intent_output = gr.Textbox(
                label="User Intent"
            )

            situation_output = gr.Textbox(
                label="Situation"
            )

            tone_output = gr.Textbox(
                label="Selected Tone"
            )

    # --------------------------------
    # BUTTON ACTION
    # --------------------------------

    analyze_button.click(

        fn=analyze_and_respond,

        inputs=user_input,

        outputs=[
            response_output,
            emotion_output,
            confidence_output,
            intent_output,
            situation_output,
            tone_output
        ]
    )

    # --------------------------------
    # EXAMPLE INPUTS
    # --------------------------------

    gr.Markdown(
        "### Try these examples"
    )

    gr.Examples(

        examples=[
            [
                "I got selected for my dream job!"
            ],

            [
                "I have an interview tomorrow "
                "and I am really nervous."
            ],

            [
                "I worked very hard but failed "
                "my exam. I feel terrible."
            ],

            [
                "This project keeps failing "
                "and I am frustrated."
            ],

            [
                "I don't understand this topic. "
                "Can you help me?"
            ],

            [
                "I am so excited about my new job!"
            ]
        ],

        inputs=user_input
    )

/tmp/ipykernel_1266/2178940882.py:29: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(


In [ ]:
demo.launch(
    share=True,
    debug=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6c8a93375338fef38e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
